# TensorCore-FPGA Demo
## PYNQ-Z1 Matrix Multiplication Accelerator

This notebook demonstrates the TensorCore systolic array accelerator on PYNQ-Z1.

### Features
- 3x3 Systolic Array (9 MAC units)
- Q16.16 Fixed-Point Arithmetic
- VPU with L2 Normalization and Softmax
- AXI4-Lite Control Interface

### Prerequisites
Copy `tensorcore.bit`, `tensorcore.hwh`, and `tensorcore/` folder to same directory as this notebook.

## 1. Setup and Overlay Loading

In [1]:
import sys
import os
import numpy as np
import time

# Add current directory to path for tensorcore package
sys.path.insert(0, os.getcwd())

from pynq import Overlay
from tensorcore.driver import TensorCoreDriver, array_to_q16_16, q16_16_to_array

print(f"NumPy version: {np.__version__}")
print(f"Working directory: {os.getcwd()}")

NumPy version: 1.21.5
Working directory: /home/xilinx/jupyter_notebooks


In [2]:
# Load overlay
print("Loading overlay...")
overlay = Overlay("tensorcore.bit")
print("Overlay loaded!")

# List available IPs
print("\nAvailable IPs:")
for ip_name in overlay.ip_dict:
    ip = overlay.ip_dict[ip_name]
    addr = ip.get('phys_addr') or ip.get('physical_address') or 'Unknown'
    print(f"  - {ip_name}: @ {addr}")

Loading overlay...


Overlay loaded!

Available IPs:
  - tensorcore_0: @ 1136656384
  - processing_system7_0: @ Unknown


In [3]:
# Initialize TensorCore driver
tc = TensorCoreDriver(overlay)
print(f"TensorCore initialized: {tc}")
print(f"  Done: {tc.is_done}")
print(f"  Busy: {tc.is_busy}")
print(f"  State: {tc.state}")

TensorCore initialized: TensorCoreDriver(status=IDLE, state=0)
  Done: False
  Busy: False
  State: 0


## 2. Quick Benchmark

In [4]:
# Run built-in benchmark
print("Running benchmark...\n")
results = tc.benchmark([2, 4, 8, 16])

print(f"{'Size':>8} | {'Cycles':>10} | {'Time @70MHz':>12}")
print("-" * 36)
for size, cycles in results.items():
    time_us = cycles / 70  # 70 MHz clock
    print(f"{size:>3}x{size:<3} | {cycles:>10,} | {time_us:>8.2f} µs")

Running benchmark...

    Size |     Cycles |  Time @70MHz
------------------------------------
  2x2   |         72 |     1.03 µs
  4x4   |        488 |     6.97 µs
  8x8   |      1,684 |    24.06 µs
 16x16  |     12,520 |   178.86 µs


## 3. Matrix Multiplication Test

In [5]:
# Test matmul API with random matrices
A = np.random.randn(4, 4).astype(np.float32)
W = np.random.randn(4, 4).astype(np.float32)

print("Input A:")
print(A.round(3))
print("\nInput W:")
print(W.round(3))

Input A:
[[-0.817  0.225  1.31   1.856]
 [-0.618  0.97   0.311  0.335]
 [ 1.779  0.579 -0.698  0.247]
 [ 0.587  0.659  1.208  0.286]]

Input W:
[[ 0.55   0.178 -0.71  -0.416]
 [-0.593 -0.581  0.116 -1.765]
 [-1.515 -0.772  0.724  1.015]
 [-0.127  0.473  1.217  1.82 ]]


In [6]:
# Run matrix multiplication
C, perf = tc.matmul(A, W)

print("Result C = A @ W:")
print(C.round(3))

print(f"\nPerformance:")
print(f"  Cycles: {perf['cycles']}")
print(f"  Mem Reads: {perf['mem_reads']}")
print(f"  Mem Writes: {perf['mem_writes']}")

Result C = A @ W:
[[-2.803 -0.409  3.813  4.65 ]
 [-1.429 -0.754  1.185 -0.529]
 [ 1.662  0.636 -1.402 -2.022]
 [-1.935 -1.076  0.882  0.339]]

Performance:
  Cycles: 488
  Mem Reads: 96
  Mem Writes: 16


## 4. Performance Scaling Analysis

In [7]:
# Analyze O(n^3) scaling
sizes = [2, 4, 8, 16]
cycles_list = [72, 488, 1684, 12520]  # From benchmark

print("Scaling Analysis:")
print(f"{'Size':>6} | {'Cycles':>8} | {'n³':>8} | {'Cycles/n³':>10}")
print("-" * 40)
for size, cycles in zip(sizes, cycles_list):
    n3 = size ** 3
    ratio = cycles / n3
    print(f"{size:>3}x{size:<2} | {cycles:>8,} | {n3:>8,} | {ratio:>10.2f}")

print("\n✓ Cycles/n³ should be roughly constant for O(n³) scaling")

Scaling Analysis:
  Size |   Cycles |       n³ |  Cycles/n³
----------------------------------------
  2x2  |       72 |        8 |       9.00
  4x4  |      488 |       64 |       7.62
  8x8  |    1,684 |      512 |       3.29
 16x16 |   12,520 |    4,096 |       3.06

✓ Cycles/n³ should be roughly constant for O(n³) scaling


## 5. VPU Operations Test

In [8]:
# Test VPU operations
print("Testing VPU operations...\n")

# Test L2 Normalization
tc.reset()
tc.configure(k1=4, k2=4, k3=4)
tc.set_vpu_operation(tc.VPU_OP_NORM)
tc.start(vpu_enable=True)
success = tc.wait_done()
print(f"VPU L2 Norm: {'PASS' if success else 'FAIL'}")
print(f"  Cycles: {tc.perf_cycles}")

# Test Softmax
tc.reset()
tc.configure(k1=4, k2=4, k3=4)
tc.set_vpu_operation(tc.VPU_OP_SOFTMAX)
tc.start(vpu_enable=True)
success = tc.wait_done()
print(f"\nVPU Softmax: {'PASS' if success else 'FAIL'}")
print(f"  Cycles: {tc.perf_cycles}")

Testing VPU operations...

VPU L2 Norm: PASS
  Cycles: 556

VPU Softmax: PASS
  Cycles: 523


## 6. Summary

In [9]:
print("="*50)
print("  TensorCore-FPGA Demo Complete")
print("="*50)
print("\nHardware Features:")
print("  - 3x3 Systolic Array (9 MACs)")
print("  - Q16.16 Fixed-Point Arithmetic")
print("  - Tiled Matrix Multiplication")
print("  - VPU: L2 Norm, Softmax")
print("  - 70 MHz Clock (timing clean)")
print("\nVerified Performance:")
print("  - 2x2:   72 cycles")
print("  - 4x4:   488 cycles")
print("  - 8x8:   1,684 cycles")
print("  - 16x16: 12,520 cycles")
print("\nFor more info: https://github.com/muditbhargava66/TensorCore-FPGA")

  TensorCore-FPGA Demo Complete

Hardware Features:
  - 3x3 Systolic Array (9 MACs)
  - Q16.16 Fixed-Point Arithmetic
  - Tiled Matrix Multiplication
  - VPU: L2 Norm, Softmax
  - 70 MHz Clock (timing clean)

Verified Performance:
  - 2x2:   72 cycles
  - 4x4:   488 cycles
  - 8x8:   1,684 cycles
  - 16x16: 12,520 cycles

For more info: https://github.com/muditbhargava66/TensorCore-FPGA
